In [14]:
base = "Files/raw"

def n(path):
    return spark.read.option("header", True).csv(path).count()

print("customers :", n(f"{base}/customer_master/customer_master_export.csv"))
print("depots    :", n(f"{base}/depots/depot_master.csv"))
print("rentals   :", n(f"{base}/rentals/rentals_*.csv"))
print("billing   :", n(f"{base}/billing/billing_export.csv"))

StatementMeta(, dcdb7cad-f838-459c-aa77-7b56b3e14b1b, 16, Finished, Available, Finished, False)

customers : 602
depots    : 6
rentals   : 976
billing   : 787


In [15]:
from pyspark.sql.functions import current_timestamp, input_file_name

base = "Files/raw"

def load_bronze(source_path, table_name):
    df = spark.read.option("header", True).csv(f"{base}/{source_path}") \
        .withColumn("_ingested_at", current_timestamp()) \
        .withColumn("_source_file", input_file_name())
    
    df.write.format("delta").mode("overwrite").saveAsTable(f"bronze_{table_name}")
    print(f"Table bronze_{table_name} created successfully with {df.count()} rows.")


# Ingesting all tables into Bronze Layer
load_bronze("customer_master/customer_master_export.csv", "customers")
load_bronze("depots/depot_master.csv", "depots")
load_bronze("rentals/rentals_*.csv", "rentals")
load_bronze("billing/billing_export.csv", "billing")

StatementMeta(, dcdb7cad-f838-459c-aa77-7b56b3e14b1b, 17, Finished, Available, Finished, False)

Table bronze_customers created successfully with 602 rows.
Table bronze_depots created successfully with 6 rows.
Table bronze_rentals created successfully with 976 rows.
Table bronze_billing created successfully with 787 rows.
